# Week 1 — Weather Data Integration (Yanchen Zhou)
**目标**：下载前50大机场的 NOAA 天气数据，与航班数据 join，输出 `flights_2024_weather.parquet`

**输入**：`data/flights_2024_clean.parquet`（Yihong 产出，4.7后可用）

**输出**：`data/flights_2024_weather.parquet`

## Step 1：挂载 Drive，安装依赖

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/CIS5450/data'

!pip install -q tqdm

## Step 2：定义前50大机场列表

In [ ]:
import pandas as pd
import numpy as np
import requests
import io
import gzip
from tqdm import tqdm

TOP_50_AIRPORTS = [
    'ATL','LAX','ORD','DFW','DEN','JFK','SFO','SEA','LAS','MCO',
    'EWR','CLT','PHX','IAH','MIA','BOS','MSP','FLL','DTW','PHL',
    'LGA','BWI','SLC','DCA','SAN','MDW','TPA','HNL','PDX','STL',
    'DAL','BNA','AUS','IAD','OAK','MSY','RDU','SMF','SJC','SNA',
    'MCI','SAT','RSW','CLE','PIT','IND','CMH','OGG','ABQ','BUR'
]
print(f'共 {len(TOP_50_AIRPORTS)} 个机场')

## Step 3：下载 NOAA 气象站映射表

这个文件将 IATA 机场代码映射到 NOAA 气象站的 USAF/WBAN ID。

In [ ]:
# 下载气象站历史文件
isd_url = 'https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv'
isd_history = pd.read_csv(isd_url, low_memory=False)

print('原始字段:', isd_history.columns.tolist())
isd_history.head(3)

In [ ]:
# 筛选美国境内的站点，且 ICAO 代码以 K 开头（美国机场标准前缀）
us_stations = isd_history[
    (isd_history['CTRY'] == 'US') &
    (isd_history['ICAO'].notna()) &
    (isd_history['ICAO'].str.startswith('K', na=False))
].copy()

# IATA = ICAO 去掉开头的 K（大多数情况成立）
us_stations['IATA'] = us_stations['ICAO'].str[1:]

# 只保留我们需要的机场，选最近的气象站记录
us_stations['END'] = pd.to_datetime(us_stations['END'], errors='coerce')
station_map = (
    us_stations[us_stations['IATA'].isin(TOP_50_AIRPORTS)]
    .sort_values('END', ascending=False)
    .groupby('IATA')
    .first()
    .reset_index()[['IATA', 'USAF', 'WBAN']]
)

# USAF 和 WBAN 需要零填充
station_map['USAF'] = station_map['USAF'].astype(str).str.zfill(6)
station_map['WBAN'] = station_map['WBAN'].astype(str).str.zfill(5)
station_map['station_id'] = station_map['USAF'] + station_map['WBAN']

print(f'找到气象站映射: {len(station_map)} 个机场')
missing = set(TOP_50_AIRPORTS) - set(station_map['IATA'])
if missing:
    print(f'未找到映射的机场: {missing}')
station_map.head()

## Step 4：下载 ISD-Lite 天气数据

In [ ]:
# ISD-Lite 字段说明
ISD_COLS = [
    'year', 'month', 'day', 'hour',
    'air_temp',        # 气温 (°C × 10)
    'dew_point',       # 露点温度 (°C × 10)
    'sea_level_pres',  # 海平面气压 (hPa × 10)
    'wind_dir',        # 风向 (度)
    'wind_speed',      # 风速 (m/s × 10)
    'sky_cover',       # 云层覆盖 (0-8)
    'precip_1h',       # 1小时降水量 (mm × 10)
    'precip_6h'        # 6小时降水量 (mm × 10)
]

def download_isd_lite(usaf, wban, year=2024):
    """下载单个气象站的 ISD-Lite 数据"""
    url = f'https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/{year}/{usaf}-{wban}-{year}.gz'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with gzip.open(io.BytesIO(resp.content), 'rt') as f:
            df = pd.read_fwf(f, header=None, names=ISD_COLS)
        return df
    except:
        return None

# 下载所有机场
weather_dfs = []
for _, row in tqdm(station_map.iterrows(), total=len(station_map), desc='Downloading weather'):
    wdf = download_isd_lite(row['USAF'], row['WBAN'])
    if wdf is not None:
        wdf['IATA'] = row['IATA']
        weather_dfs.append(wdf)
    else:
        print(f'  FAILED: {row["IATA"]} ({row["USAF"]}-{row["WBAN"]})')

weather_raw = pd.concat(weather_dfs, ignore_index=True)
print(f'\n天气数据总行数: {len(weather_raw):,}')

## Step 5：清洗天气数据

In [ ]:
w = weather_raw.copy()

# ISD-Lite 用 -9999 表示缺失值，统一替换为 NaN
w = w.replace(-9999, np.nan)

# 还原实际值（原始数据 ×10 存储）
w['air_temp']       = w['air_temp'] / 10         # °C
w['dew_point']      = w['dew_point'] / 10        # °C
w['sea_level_pres'] = w['sea_level_pres'] / 10   # hPa
w['wind_speed']     = w['wind_speed'] / 10       # m/s
w['precip_1h']      = w['precip_1h'].replace(-1, 0.001) / 10  # -1 表示微量降水
w['precip_6h']      = w['precip_6h'].replace(-1, 0.001) / 10

# 构建 datetime
w['obs_time'] = pd.to_datetime(
    w[['year','month','day','hour']].astype(str).agg('-'.join, axis=1),
    format='%Y-%m-%d-%H'
)

# 对缺失天气做前向填充（同一气象站内，最多填充 3 小时）
w = w.sort_values(['IATA', 'obs_time'])
weather_cols = ['air_temp','dew_point','sea_level_pres','wind_dir','wind_speed','sky_cover','precip_1h']
w[weather_cols] = w.groupby('IATA')[weather_cols].transform(lambda x: x.ffill(limit=3))

print('天气数据清洗完成')
print(w[['IATA','obs_time'] + weather_cols].head())

## Step 6：与航班数据 Join

In [ ]:
# 加载航班数据（等 Yihong 完成后）
flights = pd.read_parquet(f'{DATA_DIR}/flights_2024_clean.parquet')
print(f'航班数据: {len(flights):,} 行')

# 构造航班计划出发时间（整点对齐，用于 join）
flights['FlightDate'] = pd.to_datetime(flights['FlightDate'])
flights['dep_hour'] = flights['CRSDepTime'] // 100
flights['dep_datetime'] = flights['FlightDate'] + pd.to_timedelta(flights['dep_hour'], unit='h')

# 准备天气 lookup 表
weather_lookup = w[['IATA', 'obs_time'] + weather_cols].copy()

# Join 出发机场天气（Origin + dep_datetime）
flights = flights.merge(
    weather_lookup.rename(columns={
        'IATA': 'Origin', 'obs_time': 'dep_datetime',
        **{c: f'origin_{c}' for c in weather_cols}
    }),
    on=['Origin', 'dep_datetime'],
    how='left'
)

# Join 到达机场天气（Dest + dep_datetime，用出发时间估算，后续可改进）
flights = flights.merge(
    weather_lookup.rename(columns={
        'IATA': 'Dest', 'obs_time': 'dep_datetime',
        **{c: f'dest_{c}' for c in weather_cols}
    }),
    on=['Dest', 'dep_datetime'],
    how='left'
)

print(f'Join 后行数: {len(flights):,}')
print(f'出发机场天气缺失率: {flights["origin_air_temp"].isna().mean()*100:.2f}%')
print(f'到达机场天气缺失率: {flights["dest_air_temp"].isna().mean()*100:.2f}%')

## Step 7：衍生天气特征

In [ ]:
# 出发机场天气标签
flights['origin_is_rain']    = (flights['origin_precip_1h'] > 0).astype(int)
flights['origin_high_wind']  = (flights['origin_wind_speed'] > 10).astype(int)  # >10 m/s
flights['origin_freezing']   = (flights['origin_air_temp'] < 0).astype(int)
flights['origin_low_vis']    = (flights['origin_sky_cover'] >= 7).astype(int)   # 接近全覆盖

# 天气严重程度综合评分
flights['origin_weather_severity'] = (
    flights['origin_is_rain'] * 1.0 +
    flights['origin_high_wind'] * 1.5 +
    flights['origin_freezing'] * 1.2 +
    flights['origin_low_vis'] * 0.8
)

# 出发/到达取最差值（瓶颈效应）
flights['worst_precip'] = flights[['origin_precip_1h', 'dest_precip_1h']].max(axis=1)
flights['worst_wind']   = flights[['origin_wind_speed', 'dest_wind_speed']].max(axis=1)

print('天气衍生特征构建完成')
print(flights[['origin_is_rain','origin_high_wind','origin_freezing','origin_weather_severity','worst_precip']].describe())

## Step 8：保存输出

In [ ]:
output_path = f'{DATA_DIR}/flights_2024_weather.parquet'
flights.to_parquet(output_path, index=False)
print(f'已保存: {output_path}')
print(f'文件大小: {os.path.getsize(output_path)/1024/1024:.1f} MB')
print(f'最终行数: {len(flights):,}')
print(f'列数: {len(flights.columns)}')